# sewage-yolo26 split file counts

Auto-locates `sewage-yolo26/` whether the notebook sits next to it (`ass3/sewage-yolo26`) or one level above.

In [ ]:
from pathlib import Path

SPLITS = ('train', 'valid', 'test')
SUBS = ('images', 'labels')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Walk a few likely roots and pick the first one that actually has the split dirs.
CANDIDATES = [
    Path('sewage-yolo26'),
    Path('ass3/sewage-yolo26'),
    Path('../sewage-yolo26'),
    Path('../ass3/sewage-yolo26'),
    Path.cwd() / 'sewage-yolo26',
]
DATASET = next(
    (p for p in CANDIDATES if all((p / s / 'images').exists() for s in SPLITS)),
    None,
)
assert DATASET is not None, (
    f'sewage-yolo26 not found. cwd={Path.cwd()}. tried: '
    + ', '.join(str(p) for p in CANDIDATES)
)
print('dataset:', DATASET.resolve())

In [ ]:
rows = []
for split in SPLITS:
    counts = {}
    for sub in SUBS:
        d = DATASET / split / sub
        if not d.exists():
            counts[sub] = None
            continue
        if sub == 'images':
            counts[sub] = sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)
        else:
            counts[sub] = sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() == '.txt')
    rows.append((split, counts['images'], counts['labels']))

total_imgs = sum(r[1] or 0 for r in rows)
width = max(len(s) for s in SPLITS)
print(f"{'split':<{width}}  {'images':>7}  {'labels':>7}  {'ratio':>6}")
print('-' * (width + 26))
for split, n_img, n_lbl in rows:
    ratio = f'{(n_img / total_imgs * 100):5.1f}%' if n_img and total_imgs else '   -  '
    print(f'{split:<{width}}  {n_img!s:>7}  {n_lbl!s:>7}  {ratio:>6}')
print('-' * (width + 26))
print(f"{'total':<{width}}  {total_imgs:>7}  {sum(r[2] or 0 for r in rows):>7}")

In [ ]:
# Sanity check: every image must have a matching label (same stem).
for split in SPLITS:
    img_dir = DATASET / split / 'images'
    lbl_dir = DATASET / split / 'labels'
    if not (img_dir.exists() and lbl_dir.exists()):
        continue
    img_stems = {p.stem for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
    lbl_stems = {p.stem for p in lbl_dir.iterdir() if p.suffix.lower() == '.txt'}
    missing_lbl = sorted(img_stems - lbl_stems)
    orphan_lbl = sorted(lbl_stems - img_stems)
    print(f'[{split}] images={len(img_stems)} labels={len(lbl_stems)} '
          f'missing_label={len(missing_lbl)} orphan_label={len(orphan_lbl)}')
    if missing_lbl[:3]:
        print('  missing_label sample:', missing_lbl[:3])
    if orphan_lbl[:3]:
        print('  orphan_label sample :', orphan_lbl[:3])